# In this notebook, we're going to cover some of of the most fundamental concepts of tensor using TensowFlow

More specifically, we're goint to cover:
* Introduction to tensors
* Getting information from tensors
* Manipulating tensors
* Tensors & Numpy
* Using @tf.function (a way to speed your regular Python functions)
* Using GPUs wiht Tensorflow (or TPUs)
* Exercises to try for yourself

## Introduction to Tensors

In [70]:
# Import Tensorflow
import tensorflow as tf
import numpy as np
print(tf.__version__)

2.10.0


In [2]:
# Create tensors with tf.constant()
scalar = tf.constant(7)
scalar

<tf.Tensor: shape=(), dtype=int32, numpy=7>

In [3]:
# Check the number of dimensions of a tensor (ndim stands for number of dimensions)
scalar.ndim

0

In [4]:
# Create a vector
vector = tf.constant([10, 10])
vector

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([10, 10])>

In [5]:
# Check the dimensions of our vector
vector.ndim

1

In [6]:
# Create a matrix (has more than 1 dimension)
matrix = tf.constant([[10, 7],
                      [7, 10]])
matrix

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 7, 10]])>

In [7]:
matrix.ndim

2

In [8]:
# Create anotehr matrix
another_matrix = tf.constant([[10., 7.],
                              [3., 2.],
                              [8., 9.]], dtype=tf.float16) # specify the data type with dtype parameter
another_matrix                     

<tf.Tensor: shape=(3, 2), dtype=float16, numpy=
array([[10.,  7.],
       [ 3.,  2.],
       [ 8.,  9.]], dtype=float16)>

In [9]:
# What's the number of dimension of antoher matrix?
another_matrix.ndim

2

In [10]:
# Let's create a tensor
tensor = tf.constant([[[1, 2, 3],
                       [4, 5, 6]],
                      [[7, 8, 9],
                       [10, 11, 12]],
                      [[13, 14, 15],
                       [16, 17, 18]]])
tensor                    

<tf.Tensor: shape=(3, 2, 3), dtype=int32, numpy=
array([[[ 1,  2,  3],
        [ 4,  5,  6]],

       [[ 7,  8,  9],
        [10, 11, 12]],

       [[13, 14, 15],
        [16, 17, 18]]])>

In [11]:
tensor.ndim

3

What we've created so far:
* Scalar: a single number
* Vector: a number with direction (e.g. wind speed and direction)
* Matrix: a 2-dimensional array of number
* Tensor: an n-dimensional array of number (when n can be any number, a 0-dimensional tensor is a scalar, 1-dimensional tensor is a vector)

### Creating tensors with `tf.Variable`

In [12]:
# Create the same tensor with tf.Variable()
changeable_tensor = tf.Variable([10, 7])
unchangeable_tensor = tf.constant([10, 7])
changeable_tensor, unchangeable_tensor

(<tf.Variable 'Variable:0' shape=(2,) dtype=int32, numpy=array([10,  7])>,
 <tf.Tensor: shape=(2,), dtype=int32, numpy=array([10,  7])>)

In [13]:
# Let's try change one of the elements in our changeable tensor
changeable_tensor[0] = 7
changeable_tensor

TypeError: 'ResourceVariable' object does not support item assignment

In [ ]:
# How about we try .assign()
changeable_tensor[0].assign(7)
changeable_tensor

<tf.Variable 'Variable:0' shape=(2,) dtype=int32, numpy=array([7, 7])>

In [ ]:
# Let's try change our unchangeable tensor
unchangeable_tensor[0].assign(7)

AttributeError: 'tensorflow.python.framework.ops.EagerTensor' object has no attribute 'assign'

🔑 **Note:** Rarely in practice will you need to decide whether to use `tf.constant` or `tf.Variable` to create tensors, as TensorFlow does this for you. However, if in doubt, use `tf.constant` and change it later if needed.

### Creating random tensors

Random tensors are tensors of some arbitrary size which contain random numbers

In [ ]:
# Create two random (but the same) tensors
random_1 = tf.random.Generator.from_seed(42) # set seed for reproducibility
random_1 = random_1.normal(shape=(3, 2))
random_2 = tf.random.Generator.from_seed(42)
random_2 = random_2.normal(shape=(3, 2))

# Are they equal?
random_1, random_2, random_1 == random_2

(<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
 array([[-0.7565803 , -0.06854702],
        [ 0.07595026, -1.2573844 ],
        [-0.23193763, -1.8107855 ]], dtype=float32)>,
 <tf.Tensor: shape=(3, 2), dtype=float32, numpy=
 array([[-0.7565803 , -0.06854702],
        [ 0.07595026, -1.2573844 ],
        [-0.23193763, -1.8107855 ]], dtype=float32)>,
 <tf.Tensor: shape=(3, 2), dtype=bool, numpy=
 array([[ True,  True],
        [ True,  True],
        [ True,  True]])>)

### Shuffle the order of elements in a tensor

In [ ]:
# Shuffle a tensor (valuable for when you want to shuffle your data so the inherent order does'nt effect learning)
not_shuffled = tf.constant([[10, 7],
                            [3, 4],
                            [2, 5]])
# Shuffle our non-shuffled tensor
tf.random.shuffle(not_shuffled)

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[ 3,  4],
       [10,  7],
       [ 2,  5]])>

In [ ]:
# Shuffle our non-shuffled tensor
tf.random.set_seed(42)
tf.random.shuffle(not_shuffled, seed=42)

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4],
       [ 2,  5]])>

🛠 **Exercise:** Read through TensorFlow documentation on random seed generation: https://www.tensorflow.org/api_docs/python/tf/random/set_seed and practice writing 5 random tensors and shuffle them.

It looks like if we want our shuffled tensors to be in the same order, we've got to use the global level random seed as well as the operation level random seed:

> Rule 4: "If both the global and the operation seed are set: Both seeds are used in conjunction to determine the random sequence."

In [ ]:
tf.random.set_seed(42) # global level random seed
tf.random.shuffle(not_shuffled, seed=42) # operation level random seed

NameError: name 'not_shuffled' is not defined

### Other ways to make tensors

In [ ]:
# Create a tensor of all ones
tf.ones([10, 7])

<tf.Tensor: shape=(10, 7), dtype=float32, numpy=
array([[1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.]], dtype=float32)>

In [ ]:
# Create a tensor of all zeros
tf.zeros(shape=(3, 4))

<tf.Tensor: shape=(3, 4), dtype=float32, numpy=
array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], dtype=float32)>

### Turn NumPy arrays into tensors

The main difference between NumPy arrays and Tensorflow tensor is that tensors can be run on a GPU (much faster for numerical computing)

In [ ]:
# You can also tun Numpy arrays into tensors
import numpy as np
numpy_A = np.arange(1, 25, dtype=np.int32) # Create a NumPy array between 1 and 25
numpy_A
# X = tf.constant(some_matrix) # captial for matrix or tensor
# y = tf.constant(vector) # non-capital for vector

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24])

In [ ]:
A = tf.constant(numpy_A, shape=(3, 8))
B = tf.constant(numpy_A)
A, B

(<tf.Tensor: shape=(3, 8), dtype=int32, numpy=
 array([[ 1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16],
        [17, 18, 19, 20, 21, 22, 23, 24]])>,
 <tf.Tensor: shape=(24,), dtype=int32, numpy=
 array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24])>)

In [ ]:
3 * 8

24

In [ ]:
A.ndim

2

### Getting information from tensors

When dealing with tensors you probably want to be aware of the following attributes:
- Shape
- Rank
- Axis or dimension
- Size

In [14]:
# Create a rank 4 tensor (4 dimensions)
rank_4_tensor = tf.zeros(shape=[2, 3, 4, 5])
rank_4_tensor

<tf.Tensor: shape=(2, 3, 4, 5), dtype=float32, numpy=
array([[[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]],


       [[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]]], dtype=float32)>

In [15]:
rank_4_tensor.shape, rank_4_tensor.ndim, tf.size(rank_4_tensor)

(TensorShape([2, 3, 4, 5]), 4, <tf.Tensor: shape=(), dtype=int32, numpy=120>)

In [16]:
2 * 3 * 4 * 5

120

In [17]:
# Get various attributes of our tensor
print("Datatype of every element:", rank_4_tensor.dtype)
print("Number of dimensions (rank):", rank_4_tensor.ndim)
print("Shape of tensor:", rank_4_tensor.shape)
print("Elements along the 0 axis:", rank_4_tensor.shape[0])
print("Elements along the last axis:", rank_4_tensor.shape[-1])
print("Total number of elements in our tensor:", tf.size(rank_4_tensor))
print("Total number of elements in our tensor:", tf.size(rank_4_tensor).numpy())

Datatype of every element: <dtype: 'float32'>
Number of dimensions (rank): 4
Shape of tensor: (2, 3, 4, 5)
Elements along the 0 axis: 2
Elements along the last axis: 5
Total number of elements in our tensor: tf.Tensor(120, shape=(), dtype=int32)
Total number of elements in our tensor: 120


In [18]:
def tensor_attributes(tensor):
    """
    Prints out the attributes of a given tensor.
    """
    print("Datatype of every element:", tensor.dtype)
    print("Number of dimensions (rank):", tensor.ndim)
    print("Shape of tensor:", tensor.shape)
    print("Elements along the 0 axis:", tensor.shape[0])
    print("Elements along the last axis:", tensor.shape[-1])
    print("Total number of elements in our tensor:", tf.size(tensor))
    print("Total number of elements in our tensor:", tf.size(tensor).numpy())

In [19]:
tensor_attributes(rank_4_tensor)

Datatype of every element: <dtype: 'float32'>
Number of dimensions (rank): 4
Shape of tensor: (2, 3, 4, 5)
Elements along the 0 axis: 2
Elements along the last axis: 5
Total number of elements in our tensor: tf.Tensor(120, shape=(), dtype=int32)
Total number of elements in our tensor: 120


### Indexing tensors

Tensors can be indexed just like Python lists.

In [20]:
some_list = [1, 2, 3, 4]
some_list[:2]

[1, 2]

In [21]:
# Get the first 2 elements of each dimension
rank_4_tensor[:2, :2, :2, :2]

<tf.Tensor: shape=(2, 2, 2, 2), dtype=float32, numpy=
array([[[[0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.]]],


       [[[0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.]]]], dtype=float32)>

In [22]:
some_list[:1]

[1]

In [23]:
rank_4_tensor.shape

TensorShape([2, 3, 4, 5])

In [24]:
# Get the first element from each dimension from each index except for the final one
rank_4_tensor[:1, :1, :1, :]

<tf.Tensor: shape=(1, 1, 1, 5), dtype=float32, numpy=array([[[[0., 0., 0., 0., 0.]]]], dtype=float32)>

In [25]:
# Create a rank 2 tensor (2 dimensions)
rank_2_tensor = tf.constant([[10, 7],
                             [3, 4]])
rank_2_tensor.shape, rank_2_tensor.ndim

(TensorShape([2, 2]), 2)

In [26]:
rank_2_tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4]])>

In [27]:
some_list, some_list[-1]

([1, 2, 3, 4], 4)

In [28]:
# Get the last item of each of row of our rank 2 tensor
rank_2_tensor[:, -1]

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([7, 4])>

In [29]:
rank_2_tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4]])>

In [30]:
# Add in extra dimennsion to our rank 2 tensor
rank_3_tensor = rank_2_tensor[..., tf.newaxis]
rank_3_tensor

<tf.Tensor: shape=(2, 2, 1), dtype=int32, numpy=
array([[[10],
        [ 7]],

       [[ 3],
        [ 4]]])>

In [31]:
# Alternative to tf.newaxis
tf.expand_dims(rank_2_tensor, axis=-1) # "-1" means expand the final axis

<tf.Tensor: shape=(2, 2, 1), dtype=int32, numpy=
array([[[10],
        [ 7]],

       [[ 3],
        [ 4]]])>

In [32]:
# Expand the 0 axis
tf.expand_dims(rank_2_tensor, axis=0) # expand the 0 axis

<tf.Tensor: shape=(1, 2, 2), dtype=int32, numpy=
array([[[10,  7],
        [ 3,  4]]])>

In [33]:
# Expand the 1 axis
tf.expand_dims(rank_2_tensor, axis=1) # expand the 1 axis

<tf.Tensor: shape=(2, 1, 2), dtype=int32, numpy=
array([[[10,  7]],

       [[ 3,  4]]])>

In [34]:
rank_2_tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4]])>

### Manipulating tensors (tensor operations)

**Basic operations**

`+`, `-`, `*`, `/`

In [35]:
# You can add values to a tensor using the addition operator
tensor = tf.constant([[10, 7], [3, 4]])
tensor + 10

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[20, 17],
       [13, 14]])>

In [36]:
# Original tensor is unchanged
tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4]])>

In [37]:
# Multiplication also works
tensor * 10

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[100,  70],
       [ 30,  40]])>

In [38]:
# Subtraction if you want
tensor - 10

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[ 0, -3],
       [-7, -6]])>

In [39]:
# We can use the tensorflow built-in function too
tf.multiply(tensor, 10)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[100,  70],
       [ 30,  40]])>

In [40]:
tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4]])>

**Matrix multiplication**

In machine learning, matrix multiplication is one of the most common tensor operations.

There are two rules our tensors (or matrices) need to fulfil if we're going to matrix multiply them:

1. The inner dimension must match
2. The resulting matrix has the shape of the outer dimension

In [41]:
# Matrix multiplication in tensorflow
print(tensor)
tf.matmul(tensor, tensor)

tf.Tensor(
[[10  7]
 [ 3  4]], shape=(2, 2), dtype=int32)


<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[121,  98],
       [ 42,  37]])>

In [42]:
tensor, tensor

(<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
 array([[10,  7],
        [ 3,  4]])>,
 <tf.Tensor: shape=(2, 2), dtype=int32, numpy=
 array([[10,  7],
        [ 3,  4]])>)

In [43]:
A = tf.constant([[1, 2, 5], [7, 2, 1], [3, 3, 3]])

B = tf.constant([[3, 5], [6, 7], [1, 8]])

A, B, tf.matmul(A, B)

(<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
 array([[1, 2, 5],
        [7, 2, 1],
        [3, 3, 3]])>,
 <tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[3, 5],
        [6, 7],
        [1, 8]])>,
 <tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[20, 59],
        [34, 57],
        [30, 60]])>)

In [44]:
# Matrix multipliaction with Python operator "@"
tensor @ tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[121,  98],
       [ 42,  37]])>

In [45]:
# Create a tensor (3, 2) tensor
X = tf.constant([[1, 2], [3, 4], [5, 6]])

# Create another (3, 2) tensor
Y = tf.constant([[7, 8], [9, 10], [11, 12]])

X, Y

(<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[1, 2],
        [3, 4],
        [5, 6]])>,
 <tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[ 7,  8],
        [ 9, 10],
        [11, 12]])>)

In [46]:
# Try to matrix multiply tensors of same shape
X @ Y

InvalidArgumentError: {{function_node __wrapped__MatMul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-incompatible: In[0]: [3,2], In[1]: [3,2] [Op:MatMul]

In [47]:
# Let's change the shape of Y
tf.reshape(Y, shape=(2, 3))

<tf.Tensor: shape=(2, 3), dtype=int32, numpy=
array([[ 7,  8,  9],
       [10, 11, 12]])>

In [48]:
X.shape, tf.reshape(Y, shape=(2, 3)).shape

(TensorShape([3, 2]), TensorShape([2, 3]))

In [49]:
# Try to matrix multiply X by reshaped Y
X @ tf.reshape(Y,shape=(2, 3))

<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
array([[ 27,  30,  33],
       [ 61,  68,  75],
       [ 95, 106, 117]])>

In [50]:
tf.matmul(X, tf.reshape(Y, shape=(2, 3)))

<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
array([[ 27,  30,  33],
       [ 61,  68,  75],
       [ 95, 106, 117]])>

In [51]:
tf.reshape(X , shape=(2, 3)).shape, Y.shape

(TensorShape([2, 3]), TensorShape([3, 2]))

In [52]:
# Try change the shape of X instead of Y

tf.matmul(tf.reshape(X, shape=(2, 3)), Y)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[ 58,  64],
       [139, 154]])>

In [53]:
# Can do the same with transpose
X, tf.transpose(X), tf.reshape(X, shape=(2, 3))

(<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[1, 2],
        [3, 4],
        [5, 6]])>,
 <tf.Tensor: shape=(2, 3), dtype=int32, numpy=
 array([[1, 3, 5],
        [2, 4, 6]])>,
 <tf.Tensor: shape=(2, 3), dtype=int32, numpy=
 array([[1, 2, 3],
        [4, 5, 6]])>)

In [54]:
# Try matrix multiplication with transpose rather than reshape
tf.matmul(tf.transpose(X), Y)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[ 89,  98],
       [116, 128]])>

📖 **Resource:** Info and example of matrix multiplication: https://www.mathsisfun.com/algebra/matrix-multiplying.html

**The dot product**

Matrix multiplication is also referred to as the dot product.

You can perfrom matrix multplication using:
* `tf.matmul()`
* `tf.tensordot()`

In [55]:
X, Y

(<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[1, 2],
        [3, 4],
        [5, 6]])>,
 <tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[ 7,  8],
        [ 9, 10],
        [11, 12]])>)

In [56]:
# Peform the dot product on X and Y (requires X or Y to be transposed)
tf.tensordot(tf.transpose(X), Y, axes=1)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[ 89,  98],
       [116, 128]])>

In [57]:
# Perform matrix multiplication between X and Y (transposed)
tf.matmul(X, tf.transpose(Y))

<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
array([[ 23,  29,  35],
       [ 53,  67,  81],
       [ 83, 105, 127]])>

In [58]:
# Perform matrix multiplication between X and Y (reshaped)
tf.matmul(X, tf.reshape(Y, shape=(2, 3)))

<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
array([[ 27,  30,  33],
       [ 61,  68,  75],
       [ 95, 106, 117]])>

In [59]:
# Check the values of Y, reshape Y, and transposed Y
print("Normal Y:")
print(Y, "\n") # "\n" is for newline

print("Y reshaped to (2, 3):")
print(tf.reshape(Y, (2, 3)), "\n")

print("Y transposed:")
print(tf.transpose(Y))

Normal Y:
tf.Tensor(
[[ 7  8]
 [ 9 10]
 [11 12]], shape=(3, 2), dtype=int32) 

Y reshaped to (2, 3):
tf.Tensor(
[[ 7  8  9]
 [10 11 12]], shape=(2, 3), dtype=int32) 

Y transposed:
tf.Tensor(
[[ 7  9 11]
 [ 8 10 12]], shape=(2, 3), dtype=int32)


Generally, when perfroming matrix multiplication on two tensors and one of the axes doesn't line up, you will transpose (rather than reshape) one of the tensor to get satisfy the matrix mulitplication rules.

### Changing the datatype of a tensor

In [60]:
# Create a new tensor with defaul data type (float32)
B = tf.constant([1.7, 7.4])
B, B.dtype

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([1.7, 7.4], dtype=float32)>,
 tf.float32)

In [61]:
C = tf.constant([7, 10])
C.dtype

tf.int32

In [62]:
# Change from float32 to float16 (reduced precision)
D = tf.cast(B, dtype=tf.float16)
D, D.dtype

(<tf.Tensor: shape=(2,), dtype=float16, numpy=array([1.7, 7.4], dtype=float16)>,
 tf.float16)

In [63]:
# Change from int32 to float32
E = tf.cast(C, dtype=tf.float32)
E, E.dtype

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([ 7., 10.], dtype=float32)>,
 tf.float32)

In [64]:
E_float16 = tf.cast(E, dtype=tf.float16)
E_float16

<tf.Tensor: shape=(2,), dtype=float16, numpy=array([ 7., 10.], dtype=float16)>

### Aggregating tensors

Aggregating tensors = condensing them from multiple values down to a smaller amount of values.

In [65]:
# Get the absolute values
D = tf.constant([-7, -10])
D

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([ -7, -10])>

In [66]:
# Get the absolute values
tf.abs(D)

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([ 7, 10])>

Let's go through the following forms of aggregation:
* Get the minimum
* Get the maximum
* Get the mean of a tensor
* Get the sum of a tensor

In [71]:
# Create a random tensor with values between 0 and 100 of size 50
E = tf.constant(np.random.randint(0, 100, size=50))
E

<tf.Tensor: shape=(50,), dtype=int32, numpy=
array([ 2, 17, 17, 98, 41, 86, 32, 84, 98,  8, 15, 40,  4, 72, 44, 27, 54,
       78, 51, 99, 55, 79, 78, 22, 95, 86, 70, 19, 27, 95, 56, 83, 77, 39,
       71, 50, 78, 33, 17, 13, 93, 98, 76,  4, 34, 36, 72, 73, 53, 59])>

In [72]:
tf.size(E), E.shape, E.ndim

(<tf.Tensor: shape=(), dtype=int32, numpy=50>, TensorShape([50]), 1)

In [73]:
# Find the minimum
tf.reduce_min(E)

<tf.Tensor: shape=(), dtype=int32, numpy=2>

In [74]:
# Find the maximum
tf.reduce_max(E)

<tf.Tensor: shape=(), dtype=int32, numpy=99>

In [75]:
# FInd the mean
tf.reduce_mean(E)

<tf.Tensor: shape=(), dtype=int32, numpy=54>

In [76]:
# Find the sum
tf.reduce_sum(E)

<tf.Tensor: shape=(), dtype=int32, numpy=2708>

🛠️ **Exercise:** With what we've just learned, find the variance and standar deviation of our `E` tensor using TensorFlow methods.

In [83]:
# To find the variance of our tensor, we need access to tensorflow_probability

import tensorflow_probability as tfp

In [82]:
# Find the variance
tfp.stats.variance(E)

<tf.Tensor: shape=(), dtype=int32, numpy=893>

In [90]:
# Find the standard deviation
tf.math.reduce_std(tf.cast(E, dtype=tf.float32))

<tf.Tensor: shape=(), dtype=float32, numpy=29.884686>

In [93]:
# Find the variance of our E tensor
tf.math.reduce_variance(tf.cast(E, dtype=tf.float32))

<tf.Tensor: shape=(), dtype=float32, numpy=893.0945>

### Find the positional and maximum and minimum



In [99]:
# Create a new tensor for finding positional minimum and maximum
tf.random.set_seed(42)
F = tf.random.uniform(shape=[50])
F

<tf.Tensor: shape=(50,), dtype=float32, numpy=
array([0.6645621 , 0.44100678, 0.3528825 , 0.46448255, 0.03366041,
       0.68467236, 0.74011743, 0.8724445 , 0.22632635, 0.22319686,
       0.3103881 , 0.7223358 , 0.13318717, 0.5480639 , 0.5746088 ,
       0.8996835 , 0.00946367, 0.5212307 , 0.6345445 , 0.1993283 ,
       0.72942245, 0.54583454, 0.10756552, 0.6767061 , 0.6602763 ,
       0.33695042, 0.60141766, 0.21062577, 0.8527372 , 0.44062173,
       0.9485276 , 0.23752594, 0.81179297, 0.5263394 , 0.494308  ,
       0.21612847, 0.8457197 , 0.8718841 , 0.3083862 , 0.6868038 ,
       0.23764038, 0.7817228 , 0.9671384 , 0.06870162, 0.79873943,
       0.66028714, 0.5871513 , 0.16461694, 0.7381023 , 0.32054043],
      dtype=float32)>

In [100]:
# Find the positional maximum
tf.argmax(F)

<tf.Tensor: shape=(), dtype=int64, numpy=42>

In [102]:
# Index on our largest value position
F[tf.argmax(F)]

<tf.Tensor: shape=(), dtype=float32, numpy=0.9671384>

In [103]:
# Find the max values of F
tf.reduce_max(F)

<tf.Tensor: shape=(), dtype=float32, numpy=0.9671384>

In [106]:
# Check for equality
F[tf.argmax(F)] == tf.reduce_max(F)

<tf.Tensor: shape=(), dtype=bool, numpy=True>

In [107]:
# Find the minimum using the positional minimum index
F[tf.argmin(F)]

<tf.Tensor: shape=(), dtype=float32, numpy=0.009463668>

In [108]:
tf.argmin(F)

<tf.Tensor: shape=(), dtype=int64, numpy=16>

In [109]:
# Check for equality
F[tf.argmin(F)] == tf.reduce_min(F)

<tf.Tensor: shape=(), dtype=bool, numpy=True>

### Squeezing a tensor (removing all single dimensions)

In [112]:
# Create a tensor to get started
tf.random.set_seed(42)
G = tf.constant(tf.random.uniform(shape=[50]), shape=(1, 1, 1, 1, 50))
G

<tf.Tensor: shape=(1, 1, 1, 1, 50), dtype=float32, numpy=
array([[[[[0.6645621 , 0.44100678, 0.3528825 , 0.46448255, 0.03366041,
           0.68467236, 0.74011743, 0.8724445 , 0.22632635, 0.22319686,
           0.3103881 , 0.7223358 , 0.13318717, 0.5480639 , 0.5746088 ,
           0.8996835 , 0.00946367, 0.5212307 , 0.6345445 , 0.1993283 ,
           0.72942245, 0.54583454, 0.10756552, 0.6767061 , 0.6602763 ,
           0.33695042, 0.60141766, 0.21062577, 0.8527372 , 0.44062173,
           0.9485276 , 0.23752594, 0.81179297, 0.5263394 , 0.494308  ,
           0.21612847, 0.8457197 , 0.8718841 , 0.3083862 , 0.6868038 ,
           0.23764038, 0.7817228 , 0.9671384 , 0.06870162, 0.79873943,
           0.66028714, 0.5871513 , 0.16461694, 0.7381023 , 0.32054043]]]]],
      dtype=float32)>

In [113]:
G.shape

TensorShape([1, 1, 1, 1, 50])

In [115]:
G_squeezed = tf.squeeze(G)
G_squeezed, G_squeezed.shape

(<tf.Tensor: shape=(50,), dtype=float32, numpy=
 array([0.6645621 , 0.44100678, 0.3528825 , 0.46448255, 0.03366041,
        0.68467236, 0.74011743, 0.8724445 , 0.22632635, 0.22319686,
        0.3103881 , 0.7223358 , 0.13318717, 0.5480639 , 0.5746088 ,
        0.8996835 , 0.00946367, 0.5212307 , 0.6345445 , 0.1993283 ,
        0.72942245, 0.54583454, 0.10756552, 0.6767061 , 0.6602763 ,
        0.33695042, 0.60141766, 0.21062577, 0.8527372 , 0.44062173,
        0.9485276 , 0.23752594, 0.81179297, 0.5263394 , 0.494308  ,
        0.21612847, 0.8457197 , 0.8718841 , 0.3083862 , 0.6868038 ,
        0.23764038, 0.7817228 , 0.9671384 , 0.06870162, 0.79873943,
        0.66028714, 0.5871513 , 0.16461694, 0.7381023 , 0.32054043],
       dtype=float32)>,
 TensorShape([50]))

### One-hot encoding tensors

In [117]:
# Create a list of indices
some_list = [0, 1, 2, 3] # could be red, green, blue, purple

# One hot encoude our list of indices
tf.one_hot(some_list, depth=4)

<tf.Tensor: shape=(4, 4), dtype=float32, numpy=
array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]], dtype=float32)>

In [118]:
# Specify custom values for one hot encoding
tf.one_hot(some_list, depth=4, on_value="yo I love deep learning", off_value="I also like to dance")

<tf.Tensor: shape=(4, 4), dtype=string, numpy=
array([[b'yo I love deep learning', b'I also like to dance',
        b'I also like to dance', b'I also like to dance'],
       [b'I also like to dance', b'yo I love deep learning',
        b'I also like to dance', b'I also like to dance'],
       [b'I also like to dance', b'I also like to dance',
        b'yo I love deep learning', b'I also like to dance'],
       [b'I also like to dance', b'I also like to dance',
        b'I also like to dance', b'yo I love deep learning']],
      dtype=object)>

### Squaring, log, square root


In [119]:
# Create a new tensor
H = tf.range(1, 10)
H

<tf.Tensor: shape=(9,), dtype=int32, numpy=array([1, 2, 3, 4, 5, 6, 7, 8, 9])>

In [120]:
# Square it
tf.square(H)

<tf.Tensor: shape=(9,), dtype=int32, numpy=array([ 1,  4,  9, 16, 25, 36, 49, 64, 81])>

In [122]:
# Find the square root
tf.sqrt(tf.cast(H, dtype=tf.float32))

<tf.Tensor: shape=(9,), dtype=float32, numpy=
array([1.       , 1.4142135, 1.7320508, 2.       , 2.236068 , 2.4494898,
       2.6457512, 2.828427 , 3.       ], dtype=float32)>

In [125]:
# Find the log
tf.math.log(tf.cast(H, dtype=tf.float32))

<tf.Tensor: shape=(9,), dtype=float32, numpy=
array([0.       , 0.6931472, 1.0986123, 1.3862944, 1.609438 , 1.7917595,
       1.9459102, 2.0794415, 2.1972246], dtype=float32)>

### Tensors and Numpy

Tensorflow interacts beautifully with NumPy arrays.

🔑**Note:** One of the main differences between a TensorFlow tensor and a NumPy array is that a TensorFlow tensor can be run on a GPU or TPU (for faster numerical processing)

In [126]:
# Create a tensor directly from a NumPy arrays
J = tf.constant(np.array([3., 7., 10.]))
J

<tf.Tensor: shape=(3,), dtype=float64, numpy=array([ 3.,  7., 10.])>

In [127]:
# Conver our tensor back to a NumPy array
np.array(J), type(np.array(J))

(array([ 3.,  7., 10.]), numpy.ndarray)

In [129]:
# Convert tensor J to a NumPy array
J.numpy(), type(J.numpy())

(array([ 3.,  7., 10.]), numpy.ndarray)

In [133]:
# The default types of each are slightly different
numpy_J = tf.constant(np.array([3., 7., 10.]))
tensor_J = tf.constant([3., 7., 10.])

# Check the datatypes of each
numpy_J.dtype, tensor_J.dtype

(tf.float64, tf.float32)

### Finding access to GPUs

In [137]:
tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

In [135]:
!nvidia-smi

Thu Sep 17 18:43:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.36                 Driver Version: 596.36         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P3             11W /   58W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

> 🔑 **Note:** If you have acces to a CUDA-enabled GPU, Tensorflow will automatically use it whenever possible.

### 🛠 00. TensorFlow Fundamentals Exercises
1. Create a vector, scalar, matrix and tensor with values of your choosing using tf.constant().
2. Find the shape, rank and size of the tensors you created in 1.
3. Create two tensors containing random values between 0 and 1 with shape [5, 300].
4. Multiply the two tensors you created in 3 using matrix multiplication.
5. Multiply the two tensors you created in 3 using dot product.
6. Create a tensor with random values between 0 and 1 with shape [224, 224, 3].
7. Find the min and max values of the tensor you created in 6 along the first axis.
8. Created a tensor with random values of shape [1, 224, 224, 3] then squeeze it to change the shape to [224, 224, 3].
9. Create a tensor with shape [10] using your own choice of values, then find the index which has the maximum value.
10. One-hot encode the tensor you created in 9.

In [144]:
scalar = tf.constant([1])
vector = tf.constant([[1, 2], [3, 4]])
matrix = tf.constant([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
tensor = tf.constant([[[[1, 2], [3, 4]], [[5, 6], [7, 8]]]])

scalar, vector, matrix, tensor

(<tf.Tensor: shape=(1,), dtype=int32, numpy=array([1])>,
 <tf.Tensor: shape=(2, 2), dtype=int32, numpy=
 array([[1, 2],
        [3, 4]])>,
 <tf.Tensor: shape=(2, 2, 2), dtype=int32, numpy=
 array([[[1, 2],
         [3, 4]],
 
        [[5, 6],
         [7, 8]]])>,
 <tf.Tensor: shape=(1, 2, 2, 2), dtype=int32, numpy=
 array([[[[1, 2],
          [3, 4]],
 
         [[5, 6],
          [7, 8]]]])>)

In [145]:
tensor_attributes(tensor)

Datatype of every element: <dtype: 'int32'>
Number of dimensions (rank): 4
Shape of tensor: (1, 2, 2, 2)
Elements along the 0 axis: 1
Elements along the last axis: 2
Total number of elements in our tensor: tf.Tensor(8, shape=(), dtype=int32)
Total number of elements in our tensor: 8


In [153]:
tf.random.set_seed(42)
rand_tensor = tf.random.uniform(shape=(5, 300))
rand_tensor

rand_tensor2 = tf.random.uniform(shape=(5, 300))

In [154]:
tf.matmul(rand_tensor, tf.transpose(rand_tensor2))

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[80.33342 , 73.40498 , 77.15965 , 73.98369 , 80.90055 ],
       [75.146355, 68.80439 , 74.24303 , 71.8418  , 75.60204 ],
       [79.7594  , 75.64456 , 77.79758 , 74.74876 , 80.55982 ],
       [75.08526 , 69.06409 , 74.30775 , 72.27616 , 76.05669 ],
       [85.05688 , 74.26627 , 78.00687 , 74.88679 , 83.134155]],
      dtype=float32)>

In [155]:
tf.tensordot(rand_tensor, tf.transpose(rand_tensor2), axes=1)

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[80.33342 , 73.40498 , 77.15965 , 73.98369 , 80.90055 ],
       [75.146355, 68.80439 , 74.24303 , 71.8418  , 75.60204 ],
       [79.7594  , 75.64456 , 77.79758 , 74.74876 , 80.55982 ],
       [75.08526 , 69.06409 , 74.30775 , 72.27616 , 76.05669 ],
       [85.05688 , 74.26627 , 78.00687 , 74.88679 , 83.134155]],
      dtype=float32)>

In [156]:
random = tf.random.normal(shape=[224, 224, 3])
random

<tf.Tensor: shape=(224, 224, 3), dtype=float32, numpy=
array([[[-0.55909735, -0.5347214 ,  2.3730333 ],
        [-1.5725931 ,  0.8055056 , -0.83387697],
        [ 0.30611223,  2.2660494 ,  0.2856414 ],
        ...,
        [ 0.7827292 ,  1.1971072 ,  0.38341054],
        [ 0.03752969, -0.9613425 , -0.8092496 ],
        [ 2.7131703 , -1.6265521 ,  1.2932913 ]],

       [[-0.6823449 ,  0.14214388,  0.50705355],
        [ 0.00296619,  0.9580631 ,  1.1646736 ],
        [-1.0783527 , -0.24004726,  0.54039246],
        ...,
        [ 0.04354762, -1.6184464 ,  2.3399143 ],
        [-1.4524025 , -2.2181118 ,  0.11643994],
        [-0.10530236, -0.5806545 , -1.3687057 ]],

       [[-1.9031777 , -0.25401774, -1.4243866 ],
        [-0.38556483, -0.66090894,  1.7656883 ],
        [-1.1690418 , -1.7929335 ,  0.5207905 ],
        ...,
        [-0.06442755,  1.5604333 , -1.4724054 ],
        [ 0.23628023,  0.6665745 ,  1.2343851 ],
        [-0.03816089, -0.5820887 ,  0.33958212]],

       ...,

     

In [158]:
tf.argmax(random, axis=0), tf.argmin(random, axis=0)

(<tf.Tensor: shape=(224, 3), dtype=int64, numpy=
 array([[ 99, 100,  25],
        [145, 124, 132],
        [219, 147, 220],
        [106,  22, 132],
        [134, 176, 122],
        [102,  55, 166],
        [ 13,  75, 155],
        [111,  72,  47],
        [107,  85, 170],
        [157, 200, 131],
        [ 14,  71,  92],
        [ 29, 208, 185],
        [ 84,   5, 151],
        [ 96,  24,  76],
        [188, 165,  64],
        [ 94, 205,  41],
        [  2,  41, 181],
        [ 30,  97, 180],
        [ 63, 193,   1],
        [ 88, 113, 160],
        [ 48,  90,  26],
        [202,   4, 132],
        [  3,  80, 130],
        [222, 219, 186],
        [142, 195, 188],
        [ 86, 116,  61],
        [198, 210,  74],
        [ 43, 121, 215],
        [186, 168, 217],
        [135,   1,  30],
        [ 90, 131,  41],
        [ 14, 214, 179],
        [149, 146, 130],
        [199,  67, 115],
        [ 83, 129, 155],
        [108,  52,   2],
        [ 76, 152, 125],
        [157,  68, 176],
 

In [160]:
random = tf.random.normal(shape=[1, 224, 224, 3])

tf.squeeze(random)

<tf.Tensor: shape=(224, 224, 3), dtype=float32, numpy=
array([[[ 0.00924649, -0.66206276, -0.7410269 ],
        [ 1.1985261 ,  0.8636208 ,  0.39257562],
        [ 0.93457496, -0.16017465, -2.1050534 ],
        ...,
        [ 1.4473414 ,  1.2453167 , -0.05608366],
        [ 0.48070228,  0.21038947,  0.5693352 ],
        [ 1.1329907 , -0.10074199, -0.42629182]],

       [[-1.001762  , -0.41300747,  1.9709021 ],
        [ 1.1261594 ,  0.80304456, -0.13455918],
        [ 1.4553814 , -0.7077329 , -0.14673625],
        ...,
        [-1.5051513 , -0.86866486,  0.79102683],
        [ 0.9567317 ,  0.9247524 ,  1.040178  ],
        [-0.03240188, -0.126545  ,  1.3657709 ]],

       [[ 0.61522555,  0.21104665,  0.92734694],
        [ 1.0903655 ,  2.8252935 , -2.4538724 ],
        [ 0.7269391 , -0.90018594, -0.48686114],
        ...,
        [ 0.27206483,  0.39592102,  0.13167113],
        [ 0.41388002, -0.5150849 , -0.96912825],
        [ 1.2204841 ,  1.5515296 , -0.14588062]],

       ...,

     

In [163]:
shape_10 = tf.constant([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

shape_10, tf.argmax(shape_10)

(<tf.Tensor: shape=(10,), dtype=int32, numpy=array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])>,
 <tf.Tensor: shape=(), dtype=int64, numpy=9>)

In [164]:
tf.one_hot(shape_10, depth=10)

<tf.Tensor: shape=(10, 10), dtype=float32, numpy=
array([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)>

## TensorFlow 2 quickstart for beginners

In [167]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.10.0


In [175]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

In [176]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10),
])

In [177]:
predictions = model(x_train[:1]).numpy()
predictions

array([[ 0.3152539 ,  0.3402536 , -0.3748713 ,  0.1058647 , -0.7734156 ,
         0.21548468,  0.22738656,  0.15247315,  0.23641707,  0.0556064 ]],
      dtype=float32)

In [178]:
tf.nn.softmax(predictions).numpy()

array([[0.12436586, 0.12751415, 0.06237113, 0.10087059, 0.04186952,
        0.11255685, 0.1139045 , 0.1056833 , 0.11493777, 0.0959263 ]],
      dtype=float32)

In [179]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [180]:
loss_fn(y_train[:1], predictions).numpy()

2.1842968

In [182]:
model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
model

In [183]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 [==============================] - 2s 902us/step - loss: 0.3012 - accuracy: 0.9127
Epoch 2/5
1875/1875 [==============================] - 2s 840us/step - loss: 0.1449 - accuracy: 0.9570
Epoch 3/5
1875/1875 [==============================] - 2s 864us/step - loss: 0.1083 - accuracy: 0.9676
Epoch 4/5
1875/1875 [==============================] - 2s 896us/step - loss: 0.0884 - accuracy: 0.9724
Epoch 5/5
1875/1875 [==============================] - 2s 944us/step - loss: 0.0761 - accuracy: 0.9757


In [184]:
model.evaluate(x_test,  y_test, verbose=2)

313/313 - 0s - loss: 0.0761 - accuracy: 0.9759 - 316ms/epoch - 1ms/step


[0.0760616734623909, 0.9758999943733215]

In [187]:
probability_model = tf.keras.Sequential([
    model,
    tf.keras.layers.Softmax()
])

In [188]:
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[1.9270933e-08, 3.6068275e-08, 2.8311328e-05, 1.3681745e-05,
        7.5136165e-12, 2.9210593e-08, 8.9080761e-14, 9.9995756e-01,
        1.9804199e-08, 1.8263050e-07],
       [6.7426214e-08, 1.7286617e-06, 9.9999595e-01, 1.8687565e-06,
        1.8029768e-15, 2.3116183e-07, 3.2458377e-08, 2.6687003e-12,
        1.0522031e-07, 3.4934007e-14],
       [5.5296221e-07, 9.9492407e-01, 3.6248211e-03, 3.2514756e-05,
        6.0937269e-05, 1.2725677e-05, 8.6160828e-05, 8.7944028e-04,
        3.6370358e-04, 1.5086598e-05],
       [9.9979192e-01, 2.5789122e-09, 1.5790631e-04, 4.9939263e-08,
        2.0320247e-06, 6.9489073e-07, 4.3210825e-05, 3.5125374e-06,
        1.0808557e-07, 6.5387167e-07],
       [4.5431502e-06, 2.0433191e-09, 1.0626502e-05, 4.0035519e-07,
        9.9650371e-01, 5.9297349e-07, 1.4378186e-06, 8.9576301e-05,
        2.7487761e-07, 3.3888328e-03]], dtype=float32)>